# LeetCode 921 — Minimum Add to Make Parentheses Valid bằng Python

Link đề: https://leetcode.com/problems/minimum-add-to-make-parentheses-valid/ · Độ khó: **Medium** · Chủ đề: String, Stack, Greedy, Parentheses

Skeleton LeetCode:

```python
class Solution:
    def minAddToMakeValid(self, s: str) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: duyệt một lượt, giữ số `'('` **đang chờ** được đóng (`open_`) và số `')'` **thừa** không có `'('` để ghép (`extra_close`); đáp án `= open_ + extra_close`. Oracle: **vét cạn mọi chuỗi hợp lệ** chứa `s` như một dãy con (chèn ký tự = mở rộng thành siêu chuỗi) — tìm độ dài siêu chuỗi hợp lệ ngắn nhất.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một chuỗi ngoặc hợp lệ khi rỗng, hoặc `AB`, hoặc `(A)`; mỗi lần được **chèn** một ngoặc ở vị trí bất kỳ; tìm số lần chèn **ít nhất** để `s` hợp lệ.
2. Nắm công cụ Python: duyệt ký tự, biến đếm số dư (balance), `min` của số dư tiền tố, `str.replace` lặp, `list` làm ngăn xếp.
3. Đi từ **ngăn xếp ghép cặp** → **hai biến đếm** → **công thức từ số dư tiền tố nhỏ nhất** → **xoá lặp `"()"`**.
4. Hiểu mấu chốt: mỗi `')'` không ghép được với `'('` nào trước nó **bắt buộc** cần thêm một `'('`; mỗi `'('` còn dư cuối cùng **bắt buộc** cần thêm một `')'`; hai loại này độc lập và cộng lại là tối thiểu.
5. Tránh lỗi kinh điển: dùng `|đếm '(' − đếm ')'|` (bỏ qua thứ tự, ví dụ `")("` ra `0` thay vì `2`), pop ngăn xếp rỗng, chỉ đếm `')'` thừa mà quên `'('` dư (hoặc ngược lại), và coi số dư âm là "đã bù" cho các `'('` sau.

## 2. Đề bài & ví dụ

Chuỗi ngoặc **hợp lệ** khi và chỉ khi:

- nó là chuỗi rỗng, hoặc
- nó viết được dạng `AB` (nối `A` và `B`) với `A`, `B` hợp lệ, hoặc
- nó viết được dạng `(A)` với `A` hợp lệ.

Cho chuỗi ngoặc `s`. Trong một nước đi, ta được **chèn** một ngoặc (mở hoặc đóng) vào **vị trí bất kỳ** của chuỗi.

Trả về số nước đi **tối thiểu** để `s` trở nên hợp lệ.

**Ví dụ 1:** `s = "())"` → `1` (chèn `'('` ở đầu: `"(())"`).

**Ví dụ 2:** `s = "((("` → `3` (chèn ba `')'` ở cuối).

**Ràng buộc:** `1 <= s.length <= 1000`; `s[i]` là `'('` hoặc `')'`.

In [1]:
import itertools
import random
import time
from typing import List

examples = [("())", 1), ("(((", 3)]
for s, want in examples:
    print(f"s = {s!r} -> {want}")

s = '())' -> 1
s = '(((' -> 3


## 3. Kiến thức nền

### 3.1 Chuỗi hợp lệ và số dư (balance)

Đặt `bal` = số `'('` trừ số `')'` khi duyệt. Chuỗi hợp lệ ⇔ (i) `bal ≥ 0` ở **mọi tiền tố** và (ii) `bal = 0` ở cuối. Điều kiện (i) nói không có `')'` nào "đóng quá"; (ii) nói không còn `'('` nào chưa được đóng.

In [2]:
def is_valid(s):
    bal = 0
    for ch in s:
        bal += 1 if ch == "(" else -1
        if bal < 0:
            return False
    return bal == 0


print([(t, is_valid(t)) for t in ["", "()", "(())", "()()", ")(", "(()", "())"]])

[('', True), ('()', True), ('(())', True), ('()()', True), (')(', False), ('(()', False), ('())', False)]


### 3.2 Hai loại "thiếu"

Duyệt từ trái sang phải, ghép mỗi `')'` với một `'('` còn chờ (nếu có). Hai trường hợp:

- `')'` khi **không có `'('` đang chờ**: nó "thừa" ⇒ phải chèn một `'('` đứng trước nó (một phép chèn).
- `'('` còn **chờ đến cuối**: phải chèn một `')'` đứng sau nó (một phép chèn).

Mỗi phép chèn khử được **tối đa một** "thiếu" nên đáp án `≥` tổng hai loại; và tổng đó đủ.

### 3.3 Số dư tiền tố nhỏ nhất

`m = min(0, min_k bal_k)` là mức "âm sâu nhất" của số dư. Cần chèn `−m` dấu `'('` ở đầu để nâng mức nhỏ nhất lên `0`; khi đó số dư cuối là `bal_n − m`, cần thêm `bal_n − m` dấu `')'` ở cuối. Tổng `= −m + (bal_n − m)`.

### 3.4 Xoá lặp `"()"`

Mỗi lần xoá một cặp `"()"` liền nhau ta bỏ đi một cặp đã ghép. Khi không còn `"()"`, chuỗi còn lại có dạng `")))…((("` (một khối `')'` rồi một khối `'('`); độ dài của nó chính là số nước chèn tối thiểu.

In [3]:
s = "())(()("
t = s
steps = [t]
while "()" in t:
    t = t.replace("()", "", 1)
    steps.append(t)
print(" -> ".join(repr(x) for x in steps), "| còn lại:", repr(t), "độ dài", len(t))

'())(()(' -> ')(()(' -> ')((' | còn lại: ')((' độ dài 3


## 4. Ý tưởng nền tảng / trực giác

`s = "())(()("`. Duyệt và theo dõi `open_` (số `'('` chờ) và `extra` (số `')'` thừa):

| ký tự | `open_` | `extra` | ghi chú |
|---|---|---|---|
| `(` | 1 | 0 | chờ một `)` |
| `)` | 0 | 0 | ghép |
| `)` | 0 | 1 | thừa: cần chèn `(` |
| `(` | 1 | 1 | |
| `(` | 2 | 1 | |
| `)` | 1 | 1 | ghép |
| `(` | 2 | 1 | |

Kết quả `open_ + extra = 2 + 1 = 3`: chèn `(` trước `)` thừa và hai `)` ở cuối.

In [4]:
open_ = extra = 0
for ch in "())(()(":
    if ch == "(":
        open_ += 1
    elif open_ > 0:
        open_ -= 1                                       # ghép với một '(' đang chờ
    else:
        extra += 1                                       # ')' không có '(' để ghép
    print(f"{ch}: open_={open_}, extra={extra}")
print("đáp án =", open_ + extra)

(: open_=1, extra=0
): open_=0, extra=0
): open_=0, extra=1
(: open_=1, extra=1
(: open_=2, extra=1
): open_=1, extra=1
(: open_=2, extra=1
đáp án = 3


## 5. Cách 1 — Ngăn xếp ghép cặp

**Ý tưởng:** duyệt ký tự; `'('` đẩy vào ngăn xếp; `')'`: nếu ngăn xếp có `'('` ở đỉnh thì pop (ghép), ngược lại đẩy `')'` vào (chưa ghép được). Cuối cùng kích thước ngăn xếp = số ký tự **chưa ghép** = số phép chèn cần.

In [5]:
def add_stack(s):
    stack = []
    for ch in s:
        if ch == ")" and stack and stack[-1] == "(":
            stack.pop()                                  # ghép "()"
        else:
            stack.append(ch)                             # '(' hoặc ')' chưa ghép được
    return len(stack)


for s, want in examples:
    assert add_stack(s) == want
print([add_stack(s) for s, _ in examples])

[1, 3]


In [6]:
# Code nộp LeetCode cho Cách 1
class SolutionStack:
    def minAddToMakeValid(self, s: str) -> int:
        stack = []
        for ch in s:
            if ch == ")" and stack and stack[-1] == "(":
                stack.pop()
            else:
                stack.append(ch)
        return len(stack)

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

**Nút thắt:** ngăn xếp chỉ chứa ngoặc; khi ngăn xếp rỗng ta chỉ cần biết nó **có** `'('` hay không. Hai bộ đếm là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

Ngăn xếp luôn có dạng `)))…(((`: không bao giờ có `'('` rồi `')'` (vì `"()"` sẽ bị ghép). Vậy chỉ cần hai số: số `')'` thừa (`extra`) và số `'('` đang chờ (`open_`). Một lượt `O(n)`, bộ nhớ `O(1)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** hai bộ đếm, **Cách 3** công thức từ số dư tiền tố nhỏ nhất, **Cách 4** xoá lặp `"()"`.

### Cách 2 — Hai bộ đếm

**Giải thích từng đoạn:**

1. `open_ = 0` (số `'('` đang chờ), `extra = 0` (số `')'` thừa).
2. `'('`: `open_ += 1`. `')'`: nếu `open_ > 0` thì `open_ -= 1`, ngược lại `extra += 1`.
3. Trả `open_ + extra`.

In [7]:
def add_counters(s):
    open_ = extra = 0
    for ch in s:
        if ch == "(":
            open_ += 1
        elif open_:
            open_ -= 1
        else:
            extra += 1
    return open_ + extra


for s, want in examples:
    assert add_counters(s) == want
print([add_counters(s) for s, _ in examples])

[1, 3]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionCounters:
    def minAddToMakeValid(self, s: str) -> int:
        open_ = extra = 0
        for ch in s:
            if ch == "(":
                open_ += 1
            elif open_:
                open_ -= 1
            else:
                extra += 1
        return open_ + extra

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Công thức từ số dư tiền tố nhỏ nhất

**Ý tưởng:** tính `bal` tiền tố và `lowest = min(0, min bal)`. Cần `−lowest` dấu `'('` thêm ở đầu và `bal_n − lowest` dấu `')'` thêm ở cuối. Đáp án `= (−lowest) + (bal_n − lowest) = bal_n − 2·lowest`.

In [9]:
def add_formula(s):
    bal = lowest = 0
    for ch in s:
        bal += 1 if ch == "(" else -1
        lowest = min(lowest, bal)                        # mức âm sâu nhất của số dư
    return bal - 2 * lowest                              # (-lowest) dấu '(' ở đầu + (bal - lowest) dấu ')' ở cuối


for s, want in examples:
    assert add_formula(s) == want
print([add_formula(s) for s, _ in examples])

[1, 3]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionFormula:
    def minAddToMakeValid(self, s: str) -> int:
        bal = lowest = 0
        for ch in s:
            bal += 1 if ch == "(" else -1
            lowest = min(lowest, bal)
        return bal - 2 * lowest

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ. Đây là cách cho thấy đáp án phụ thuộc đúng hai đại lượng: số dư cuối và số dư tiền tố nhỏ nhất.

### Cách 4 — Xoá lặp `"()"`

**Ý tưởng:** lặp: nếu `s` chứa `"()"` thì xoá một lần xuất hiện; kết thúc khi không còn. Độ dài còn lại là đáp án (chuỗi còn lại có dạng `)…)(…(`).

In [11]:
def add_replace(s):
    while "()" in s:
        s = s.replace("()", "")                          # xoá mọi cặp liền nhau trong một lượt, lặp lại cho cặp lồng
    return len(s)


for s, want in examples:
    assert add_replace(s) == want
print([add_replace(s) for s, _ in examples])

[1, 3]


In [12]:
# Code nộp LeetCode cho Cách 4 (O(n^2) trong trường hợp xấu: "((((...))))")
class SolutionReplace:
    def minAddToMakeValid(self, s: str) -> int:
        while "()" in s:
            s = s.replace("()", "")
        return len(s)

**Độ phức tạp:** mỗi vòng xoá ít nhất một cặp; chuỗi lồng sâu `"((((…))))"` cần `n/2` vòng, mỗi vòng `O(n)` ⇒ `O(n²)`. Với `n ≤ 1000` vẫn nhanh, nhưng kém hơn các cách trước.

## 8. TRACE từng bước

**Trace Cách 2 và Cách 3** với `s = "())(()("`:

| ký tự | Cách 2: `open_`, `extra` | Cách 3: `bal`, `lowest` |
|---|---|---|
| `(` | 1, 0 | 1, 0 |
| `)` | 0, 0 | 0, 0 |
| `)` | 0, 1 | −1, −1 |
| `(` | 1, 1 | 0, −1 |
| `(` | 2, 1 | 1, −1 |
| `)` | 1, 1 | 0, −1 |
| `(` | 2, 1 | 1, −1 |

Cách 2: `2 + 1 = 3`. Cách 3: `bal − 2·lowest = 1 + 2 = 3` ✓. Hàm trace:

In [13]:
def trace_both(s):
    open_ = extra = bal = lowest = 0
    print(f"s = {s!r}")
    print(f"{'ký tự':>6} {'open_':>6} {'extra':>6} {'bal':>5} {'lowest':>7}")
    for ch in s:
        if ch == "(":
            open_ += 1
            bal += 1
        else:
            if open_:
                open_ -= 1
            else:
                extra += 1
            bal -= 1
        lowest = min(lowest, bal)
        print(f"{ch:>6} {open_:>6} {extra:>6} {bal:>5} {lowest:>7}")
    print(f"cách 2: {open_} + {extra} = {open_ + extra} | cách 3: {bal} - 2*({lowest}) = {bal - 2 * lowest}")
    assert open_ + extra == bal - 2 * lowest
    return open_ + extra


assert trace_both("())(()(") == 3

s = '())(()('
 ký tự  open_  extra   bal  lowest
     (      1      0     1       0
     )      0      0     0       0
     )      0      1    -1      -1
     (      1      1     0      -1
     (      2      1     1      -1
     )      1      1     0      -1
     (      2      1     1      -1
cách 2: 2 + 1 = 3 | cách 3: 1 - 2*(-1) = 3


## 9. Code nộp LeetCode

Chọn **Cách 2** (hai bộ đếm, `O(1)` bộ nhớ). Giữ nguyên `class Solution`, `minAddToMakeValid(self, s)`:

In [14]:
class Solution:
    def minAddToMakeValid(self, s: str) -> int:
        open_ = 0                                         # số '(' đang chờ được đóng
        extra = 0                                         # số ')' thừa (không có '(' để ghép) -> cần chèn '('
        for ch in s:
            if ch == "(":
                open_ += 1
            elif open_:
                open_ -= 1                                # ghép với một '(' đang chờ
            else:
                extra += 1
        return open_ + extra                              # '(' còn dư cần ')' ở cuối + ')' thừa cần '(' ở đầu


print([Solution().minAddToMakeValid(s) for s, _ in examples])      # [1, 3]
assert all(Solution().minAddToMakeValid(s) == want for s, want in examples)

[1, 3]


## 10. Vì sao đúng

**(a) Cận dưới.** Gọi `a` là số `'('` được chèn và `b` là số `')'` được chèn. Chuỗi mới hợp lệ cần (i) số dư tiền tố `≥ 0` ở mọi nơi và (ii) số dư cuối `= 0`.

- (i) Số dư tiền tố của chuỗi gốc đạt `lowest < 0` tại một vị trí `p`. Trong chuỗi mới, số dư tại vị trí tương ứng bằng `lowest + (số '(' chèn trước p) − (số ')' chèn trước p) ≤ lowest + a`. Để nó `≥ 0` cần `a ≥ −lowest`.
- (ii) Số dư cuối của chuỗi mới `= bal_n + a − b = 0` ⇒ `b = bal_n + a`.

Vậy tổng số phép chèn `a + b = 2a + bal_n ≥ 2·(−lowest) + bal_n = bal_n − 2·lowest`.

**(b) Cận trên (cấu trúc).** Chèn đúng `−lowest` dấu `'('` ở **đầu** chuỗi và `bal_n − lowest` dấu `')'` ở **cuối**: số dư tiền tố mới `= số dư cũ + (−lowest) ≥ 0` ở mọi tiền tố (vì `số dư cũ ≥ lowest`), và số dư cuối `= bal_n − lowest + ... = 0`; phần `')'` chèn ở cuối nâng nó về `0` mà không làm số dư tiền tố nào âm (chúng ở cuối, số dư giảm dần tới đúng `0`). Vậy chuỗi mới hợp lệ với `bal_n − 2·lowest` phép chèn.

**(c) Hai bộ đếm = công thức.** Tại mỗi thời điểm, `open_ = bal − lowest` (số `'('` chờ) và `extra = −lowest` (số `')'` thừa tích luỹ). Chứng minh quy nạp: thêm `'('` tăng `bal` và `open_` lên `1`, `lowest` không đổi; thêm `')'` khi `open_ > 0`: `bal` giảm 1, `lowest` không đổi (vì `bal − 1 ≥ lowest`), `open_` giảm 1 ✓; thêm `')'` khi `open_ = 0` (⇔ `bal = lowest`): `bal` giảm 1, `lowest` giảm 1 ⇒ `extra` tăng 1, `open_` vẫn `0` ✓. Do đó `open_ + extra = bal − lowest − lowest = bal − 2·lowest`.

**(d) Ngăn xếp và xoá `"()"`.** Ngăn xếp (Cách 1) giữ đúng các ngoặc chưa ghép, có dạng `)^e (^o`, nên kích thước `= e + o = extra + open_`. Cách 4 xoá cùng các cặp, còn lại đúng chuỗi đó.

Ta kiểm chứng bằng máy: oracle **vét cạn siêu chuỗi hợp lệ ngắn nhất** (mọi chuỗi hợp lệ chứa `s` như dãy con) khớp mọi cách trên mọi chuỗi độ dài `≤ 8` (mục 12D) và ngẫu nhiên (12E).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Ngăn xếp | `O(n)` | `O(n)` | trực quan |
| 2. Hai bộ đếm | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Công thức số dư | `O(n)` | `O(1)` | cho thấy bản chất |
| 4. Xoá lặp `"()"` | `O(n²)` | `O(n)` | đơn giản nhưng chậm |

Đo thực tế (`n = 1000`, và `n = 10^5` cho các cách tuyến tính):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(921)
datasets = {
    "ngẫu nhiên n=1000": "".join(rng.choice("()") for _ in range(1000)),
    "lồng sâu n=1000": "(" * 500 + ")" * 500,
    "toàn '(' n=1000": "(" * 1000,
}
for label, s in datasets.items():
    row = [f"{name} {bench(fn, s) * 1e6:9.1f} us" for name, fn in
           [("stack", add_stack), ("counters", add_counters), ("formula", add_formula), ("replace", add_replace)]]
    print(f"{label:20s}: " + " | ".join(row))
long_s = "".join(rng.choice("()") for _ in range(10 ** 5))
print("n = 10^5:", " | ".join(f"{name} {bench(fn, long_s) * 1e3:7.2f} ms" for name, fn in [("stack", add_stack), ("counters", add_counters), ("formula", add_formula)]))

ngẫu nhiên n=1000   : stack      44.5 us | counters      35.8 us | formula      59.7 us | replace      16.1 us
lồng sâu n=1000     : stack      39.0 us | counters      33.4 us | formula      49.9 us | replace    1268.1 us
toàn '(' n=1000     : stack      34.2 us | counters      27.1 us | formula      48.0 us | replace       3.0 us
n = 10^5: stack    4.75 ms | counters    4.64 ms | formula    5.87 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi chuỗi độ dài `≤ 8` với oracle siêu chuỗi hợp lệ, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
_valid_cache = {0: [""]}


def valid_strings(length):
    """Mọi chuỗi ngoặc hợp lệ độ dài `length` (chẵn), sinh theo định nghĩa: rỗng | "(" A ")" B."""
    if length in _valid_cache:
        return _valid_cache[length]
    out = []
    if length % 2 == 0 and length > 0:
        for inner in range(0, length - 1, 2):
            for a in valid_strings(inner):
                for b in valid_strings(length - 2 - inner):
                    out.append("(" + a + ")" + b)
    _valid_cache[length] = out
    return out


def is_subsequence(s, t):
    it = iter(t)
    return all(ch in it for ch in s)                     # s là dãy con của t


def oracle(s):
    """Số phép chèn tối thiểu = độ dài siêu chuỗi hợp lệ ngắn nhất chứa s như dãy con, trừ len(s).
    (Chèn các ký tự vào s cho ra chuỗi t chứa s như dãy con; ngược lại mọi t như vậy đạt được bằng len(t) - len(s) phép chèn.)"""
    n = len(s)
    for L in range(n, 2 * n + 1):
        if L % 2:
            continue
        for t in valid_strings(L):
            if is_subsequence(s, t):
                return L - n
    raise AssertionError("không thể tới đây: luôn tồn tại siêu chuỗi dài <= 2n")


IMPLS = {
    "stack": add_stack,
    "counters": add_counters,
    "formula": add_formula,
    "replace": add_replace,
    "Solution": lambda s: Solution().minAddToMakeValid(s),
    "SolutionStack": lambda s: SolutionStack().minAddToMakeValid(s),
    "SolutionCounters": lambda s: SolutionCounters().minAddToMakeValid(s),
    "SolutionFormula": lambda s: SolutionFormula().minAddToMakeValid(s),
    "SolutionReplace": lambda s: SolutionReplace().minAddToMakeValid(s),
}


def check(s, want=None, names=None):
    if want is None:
        want = oracle(s)
    for name in (names or IMPLS):
        got = IMPLS[name](s)
        assert got == want, (name, s, got, want)


for s, want in examples:
    assert oracle(s) == want
    check(s, want)
print("ví dụ đề: oracle siêu chuỗi hợp lệ và", len(IMPLS), "cài đặt đúng; số chuỗi hợp lệ độ dài 2,4,6,8:", [len(valid_strings(k)) for k in (2, 4, 6, 8)])

ví dụ đề: oracle siêu chuỗi hợp lệ và 9 cài đặt đúng; số chuỗi hợp lệ độ dài 2,4,6,8: [1, 2, 5, 14]


### B. Edge case

- `"()"` đã hợp lệ: `0`.
- Một ký tự: `"("` hoặc `")"` → `1`.
- `")("` → `2` (không thể ghép dù số lượng bằng nhau).
- Toàn `'('`/toàn `')'`.
- `")))((("` → `6`.
- `"()()()"`, `"(())"` hợp lệ.
- `"(()"` → `1`, `"())"` → `1`.
- Khối `')'` rồi khối `'('` xen kẽ với cặp hợp lệ ở giữa.

In [17]:
cases = [
    ("()", 0), ("(", 1), (")", 1), (")(", 2), ("(((", 3), (")))", 3), (")))(((", 6), ("()()()", 0), ("(())", 0),
    ("(()", 1), ("())", 1), ("())(()(", 3), (")()(", 2), ("((()))", 0), (")(())(", 2), ("()))((", 4),
]
for s, want in cases:
    check(s, want)
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

16 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_abs_diff(s):                                   # |số '(' - số ')'|: bỏ qua thứ tự
    return abs(s.count("(") - s.count(")"))


def bug_only_extra_close(s):                           # chỉ đếm ')' thừa
    open_ = extra = 0
    for ch in s:
        if ch == "(":
            open_ += 1
        elif open_:
            open_ -= 1
        else:
            extra += 1
    return extra


def bug_only_open(s):                                  # chỉ đếm '(' dư cuối cùng
    open_ = 0
    for ch in s:
        if ch == "(":
            open_ += 1
        elif open_:
            open_ -= 1
    return open_


def bug_negative_credit(s):                            # cho phép số dư âm "bù" cho '(' sau đó
    bal = 0
    for ch in s:
        bal += 1 if ch == "(" else -1
    return abs(bal)                                    # cùng bản chất với abs_diff nhưng tính bằng số dư


def bug_pop_empty(s):                                  # ngăn xếp: pop khi gặp ')' mà không kiểm tra rỗng -> ngoại lệ
    stack = []
    for ch in s:
        if ch == "(":
            stack.append(ch)
        else:
            stack.pop()                                # SAI: IndexError khi stack rỗng
    return len(stack)


def bug_formula_no_double(s):                          # công thức bal - lowest (thiếu hệ số 2)
    bal = lowest = 0
    for ch in s:
        bal += 1 if ch == "(" else -1
        lowest = min(lowest, bal)
    return bal - lowest


BUGS = [bug_abs_diff, bug_only_extra_close, bug_only_open, bug_pop_empty, bug_formula_no_double]
tests = ["".join(t) for L in range(1, 8) for t in itertools.product("()", repeat=L)]
for bug in BUGS:
    bad = None
    for s in tests:
        try:
            got = bug(s)
        except IndexError:
            got = "IndexError"
        if got != oracle(s):
            bad = (s, got)
            break
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên {bad[0]!r}: ra {bad[1]}, đúng {oracle(bad[0])}")

bug_abs_diff           sai trên ')(': ra 0, đúng 2
bug_only_extra_close   sai trên '(': ra 0, đúng 1
bug_only_open          sai trên ')': ra 0, đúng 1
bug_pop_empty          sai trên ')': ra IndexError, đúng 1
bug_formula_no_double  sai trên ')': ra 0, đúng 1


### D. Vét cạn mọi chuỗi nhỏ

Mọi chuỗi ngoặc độ dài `1..8` (`2 + 4 + … + 256 = 510` chuỗi). Oracle: siêu chuỗi hợp lệ ngắn nhất.

In [19]:
t0 = time.perf_counter()
cnt = 0
for L in range(1, 9):
    for tup in itertools.product("()", repeat=L):
        s = "".join(tup)
        want = oracle(s)
        for name in IMPLS:
            assert IMPLS[name](s) == want, (name, s)
        cnt += 1
print(cnt, "chuỗi: 9 cài đặt khớp oracle siêu chuỗi hợp lệ;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 510

510 chuỗi: 9 cài đặt khớp oracle siêu chuỗi hợp lệ; 0.0 s


### E. Ngẫu nhiên có seed và tính chất

Chuỗi ngẫu nhiên: độ dài `9..12` so với oracle, và dài tới `1000` so với nhau. Tính chất: (1) đáp án `≡ len(s) (mod 2)`; (2) đáp án `= 0` ⇔ `s` hợp lệ; (3) **chèn đúng theo cấu trúc** (`−lowest` dấu `'('` ở đầu và `bal − lowest` dấu `')'` ở cuối) cho chuỗi hợp lệ; (4) nối thêm `"()"` không đổi đáp án; (5) đảo chuỗi rồi đổi `(`↔`)` không đổi đáp án.

In [20]:
rng = random.Random(20240921)
swap = str.maketrans("()", ")(")
for _ in range(300):
    n = rng.randint(9, 12)
    s = "".join(rng.choice("()") for _ in range(n))
    check(s, names=("stack", "counters", "formula", "Solution"))
for _ in range(2000):
    n = rng.randint(1, 1000)
    p = rng.choice([0.2, 0.5, 0.8])
    s = "".join("(" if rng.random() < p else ")" for _ in range(n))
    res = {k: IMPLS[k](s) for k in ("stack", "counters", "formula", "Solution", "SolutionStack", "SolutionCounters", "SolutionFormula")}
    assert len(set(res.values())) == 1, (s, res)
    ans = res["counters"]
    assert ans % 2 == len(s) % 2                                                    # (1)
    assert (ans == 0) == is_valid(s)                                                # (2)
    bal = lowest = 0
    for ch in s:
        bal += 1 if ch == "(" else -1
        lowest = min(lowest, bal)
    fixed = "(" * (-lowest) + s + ")" * (bal - lowest)                              # (3) chèn theo cấu trúc
    assert is_valid(fixed) and len(fixed) - len(s) == ans
    assert Solution().minAddToMakeValid(s + "()") == ans                           # (4)
    assert Solution().minAddToMakeValid(s[::-1].translate(swap)) == ans             # (5)
    if len(s) <= 200:
        assert add_replace(s) == ans
print("300 chuỗi 9..12 khớp oracle; 2000 chuỗi tới 1000 ký tự: 7 cài đặt khớp; 5 tính chất đúng")

300 chuỗi 9..12 khớp oracle; 2000 chuỗi tới 1000 ký tự: 7 cài đặt khớp; 5 tính chất đúng


### F. Giới hạn

`n = 1000` và `n = 10^5` (các cách tuyến tính): dạng cực đoan (toàn `'('`, toàn `')'`, lồng sâu, xen kẽ `")("`).

In [21]:
cases = {
    "toàn (": ("(" * 1000, 1000),
    "toàn )": (")" * 1000, 1000),
    "lồng sâu hợp lệ": ("(" * 500 + ")" * 500, 0),
    "khối ) rồi khối (": (")" * 500 + "(" * 500, 1000),
    "xen kẽ )(": (")(" * 500, 2),
    "xen kẽ ()": ("()" * 500, 0),
    "(( rồi ) lẻ": ("((" * 250 + ")" * 499, 1),
}
for name, (s, want) in cases.items():
    for impl in ("stack", "counters", "formula", "Solution", "SolutionStack", "SolutionCounters", "SolutionFormula", "replace"):
        assert IMPLS[impl](s) == want, (name, impl)
    print(f"{name:20s}: {want}")
rng = random.Random(500)
big = "".join(rng.choice("()") for _ in range(10 ** 5))
assert add_stack(big) == add_counters(big) == add_formula(big) == Solution().minAddToMakeValid(big)
assert Solution().minAddToMakeValid(")(" * 50000) == 2
print("n = 10^5: ba cách tuyến tính khớp nhau; công thức đóng đúng")

toàn (              : 1000
toàn )              : 1000
lồng sâu hợp lệ     : 0
khối ) rồi khối (   : 1000
xen kẽ )(           : 2
xen kẽ ()           : 0
(( rồi ) lẻ         : 1
n = 10^5: ba cách tuyến tính khớp nhau; công thức đóng đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: so sánh số lượng `'('` và `')'` mà bỏ qua thứ tự

In [22]:
print('")(":', bug_abs_diff(")("), "| đúng:", Solution().minAddToMakeValid(")("))
assert bug_abs_diff(")(") == 0 and Solution().minAddToMakeValid(")(") == 2

")(": 0 | đúng: 2


**Vì sao sai:** `")("` có một `'('` và một `')'` nhưng không hợp lệ: `')'` đứng **trước** nên không ghép được. **Đúng:** theo dõi số dư theo thứ tự (số `')'` thừa ở tiền tố).

### Lỗi 2: chỉ đếm `')'` thừa (quên `'('` dư cuối)

In [23]:
print('"(((":', bug_only_extra_close("((("), "| đúng:", Solution().minAddToMakeValid("((("))
assert bug_only_extra_close("(((") == 0

"(((": 0 | đúng: 3


**Vì sao sai:** ba `'('` chưa đóng cần ba `')'`. **Đúng:** cộng `open_` còn lại ở cuối.

### Lỗi 3: chỉ đếm `'('` dư cuối

In [24]:
print('"())":', bug_only_open("())"), "| đúng:", Solution().minAddToMakeValid("())"))
assert bug_only_open("())") == 0 and Solution().minAddToMakeValid("())") == 1

"())": 0 | đúng: 1


**Vì sao sai:** `')'` thừa cũng cần một `'('`. **Đúng:** `open_ + extra`.

### Lỗi 4: `pop` ngăn xếp rỗng

In [25]:
try:
    bug_pop_empty(")(")
except IndexError as e:
    print("IndexError:", e)

IndexError: pop from empty list


**Vì sao sai:** gặp `')'` mà chưa có `'('` nào. **Đúng:** kiểm tra `stack` (hoặc `open_ > 0`) trước khi ghép.

### Lỗi 5: công thức thiếu hệ số `2`

In [26]:
print('")(":', bug_formula_no_double(")("), "| đúng:", Solution().minAddToMakeValid(")("))
assert bug_formula_no_double(")(") == 1 and Solution().minAddToMakeValid(")(") == 2

")(": 1 | đúng: 2


**Vì sao sai:** `lowest` âm tính cho phía `'('` thêm ở đầu **và** làm số dư cuối thấp hơn, tạo thêm `')'` cần chèn ở cuối: cả hai đều có mặt (`bal − 2·lowest`). **Đúng:** `bal - 2 * lowest`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "ngoặc hợp lệ, sửa với số phép tối thiểu" → **số dư tiền tố**; mỗi ngoặc không ghép được tạo đúng một phép sửa; cách đếm hai loại (thừa đóng, dư mở) là chuẩn.

**Mẹo:**

1. `bal ≥ 0` ở mọi tiền tố và `bal = 0` ở cuối ⇔ hợp lệ.
2. Hai bộ đếm thay cho ngăn xếp khi chỉ có một loại ngoặc.
3. Công thức `bal − 2·min(0, min_prefix)`.
4. Với nhiều loại ngoặc (`()[]{}`), phải dùng ngăn xếp thật.
5. Phân biệt **chèn** (bài này) với **xoá** (1249) với **hoán đổi** (1963).
6. Oracle: tìm siêu chuỗi hợp lệ ngắn nhất.

**Họ bài cùng pattern:** Valid Parentheses (20), Longest Valid Parentheses (32), Minimum Remove to Make Valid Parentheses (1249), Minimum Insertions to Balance a Parentheses String (1541), Minimum Number of Swaps to Make the String Balanced (1963), Score of Parentheses (856).

## 15. Biến thể

1. **Số ngoặc cần xoá** để hợp lệ (1249): `n − 2·(số cặp ghép được)`.
2. **Mỗi `'('` cần hai `')'` liền nhau** (1541).
3. **Số hoán đổi tối thiểu** để cân bằng khi số lượng bằng nhau (1963).
4. **Trả về chuỗi hợp lệ** sau khi chèn (cấu trúc ở mục 12E).
5. **Độ dài chuỗi con hợp lệ dài nhất** (32).

Cài biến thể 1 và 2 và đối chiếu vét cạn:

In [27]:
def min_removals(s):
    """Biến thể 1 (1249 dạng đếm): số ngoặc phải xoá để hợp lệ = số ngoặc không ghép được = đáp án bài này (xoá = chèn trên chuỗi này)."""
    return add_counters(s)


def min_removals_brute(s):
    n = len(s)
    best = n
    for mask in range(1 << n):                            # giữ lại các ký tự có bit 1
        t = "".join(s[i] for i in range(n) if mask >> i & 1)
        if is_valid(t):
            best = min(best, n - len(t))
    return best


def min_insertions_1541(s):
    """Biến thể 2: mỗi '(' cần ĐÚNG hai ')' liền nhau; số chèn tối thiểu."""
    res = need = 0                                        # need: số ')' còn cần cho các '(' đang chờ
    for ch in s:
        if ch == "(":
            if need % 2 == 1:                             # đang dang dở nửa cặp '))': phải chèn một ')' để khép trước khi mở '(' mới
                res += 1
                need -= 1
            need += 2
        else:
            need -= 1
            if need == -1:                                # ')' đầu tiên thừa: chèn một '(' và còn thiếu một ')'
                res += 1
                need = 1
    return res + need


_valid1541 = {0: [""]}


def valid_1541(pairs):
    """Chuỗi hợp lệ theo 1541 với `pairs` cặp: rỗng | "(" A "))" B."""
    if pairs in _valid1541:
        return _valid1541[pairs]
    out = []
    for inner in range(pairs):
        for a in valid_1541(inner):
            for b in valid_1541(pairs - 1 - inner):
                out.append("(" + a + "))" + b)
    _valid1541[pairs] = out
    return out


def oracle_1541(s):
    n = len(s)
    for pairs in range(0, n + 1):
        if 3 * pairs < n:
            continue
        for t in valid_1541(pairs):
            if is_subsequence(s, t):
                return 3 * pairs - n
    raise AssertionError


for L in range(1, 9):
    for tup in itertools.product("()", repeat=L):
        s = "".join(tup)
        assert min_removals(s) == min_removals_brute(s), s
        assert min_insertions_1541(s) == oracle_1541(s), s
assert min_insertions_1541("(()))") == 1 and min_insertions_1541("())") == 0 and min_insertions_1541("))())(") == 3
print("biến thể: số ngoặc phải xoá (1249 dạng đếm) và 1541 (mỗi '(' cần '))') khớp vét cạn trên mọi chuỗi độ dài <= 8")

biến thể: số ngoặc phải xoá (1249 dạng đếm) và 1541 (mỗi '(' cần '))') khớp vét cạn trên mọi chuỗi độ dài <= 8


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ngăn xếp | ghép cặp, đếm phần chưa ghép | `O(n)` | `O(n)` | trực quan, mở rộng nhiều loại ngoặc |
| 2. Hai bộ đếm | `open_` + `extra` | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Công thức số dư | `bal − 2·min(0, min_prefix)` | `O(n)` | `O(1)` | cho thấy bản chất |
| 4. Xoá lặp `"()"` | xoá tới khi hết | `O(n²)` | `O(n)` | ngắn nhưng chậm |

Cả bốn khớp oracle siêu chuỗi hợp lệ trên mọi 510 chuỗi nhỏ và khớp nhau tới `n = 10^5` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [28]:
print(Solution().minAddToMakeValid("()))(("), Solution().minAddToMakeValid("(()())"), Solution().minAddToMakeValid("))(("), min_insertions_1541("(()))"))

4 0 4 1


### Bài 2 (tính tay)

`s = "(()))(()"`. (a) Số dư tiền tố từng bước? (b) `lowest` và số dư cuối? (c) Đáp án và chuỗi hợp lệ sau khi chèn theo cấu trúc? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai bộ đếm:

In [29]:
def counters_todo(s):
    open_ = extra = 0
    for ch in s:
        if ch == "(":
            pass        # TODO 1: tăng số '(' đang chờ
        else:
            pass        # TODO 2: nếu có '(' chờ thì ghép, ngược lại tăng extra
    return 0            # TODO 3: open_ + extra


# Khi làm xong, bỏ comment để kiểm tra:
# assert [counters_todo(s) for s, _ in examples] == [1, 3]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **công thức số dư** `bal − 2·lowest`:

In [30]:
class MySolution:
    def minAddToMakeValid(self, s):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().minAddToMakeValid(s) == w for s, w in examples)

### Bài 5 (mini challenge): Minimum Number of Swaps to Make the String Balanced (1963)

Cho chuỗi `s` độ dài chẵn có đúng `n/2` ký tự `'['` và `n/2` ký tự `']'` (ở đây dùng `'('`/`')'`). Mỗi lần được **hoán đổi** hai ký tự bất kỳ. Tìm số hoán đổi tối thiểu để `s` hợp lệ. Gợi ý: số cặp "thừa" `extra` (đã tính ở Cách 2) rồi đáp án là `⌈extra / 2⌉`. Viết `min_swaps(s)` và kiểm tra bằng BFS/vét cạn.

In [31]:
def min_swaps(s):
    pass            # TODO


# Khi làm xong:
# assert min_swaps("))((") == 1 and min_swaps(")()(") == 1 and min_swaps("()()") == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: hai bộ đếm.
- Mức 2: `"()))(("`: ghép `()`; hai `)` thừa; hai `(` dư ⇒ `4`; `"(()())"` hợp lệ ⇒ `0`; `"))(("` ⇒ `4`.
- Mức 3: `4 0 4 1` (`min_insertions_1541("(()))") = 1`: thêm một `)` cho `(` thứ hai).

**Bài 2**

- Mức 1: số dư: `1, 2, 1, 0, −1, 0, 1, 0`.
- Mức 2: `lowest = −1`, số dư cuối `0`.
- Mức 3: đáp án `0 − 2·(−1) = 2`: chèn `(` ở đầu và `)` ở cuối: `"(" + s + ")"` hợp lệ.

**Bài 3**

- Mức 1: `open_ += 1`.
- Mức 2: `elif open_: open_ -= 1` else `extra += 1`.
- Mức 3: `return open_ + extra`.

**Bài 4**

- Mức 1: `bal`, `lowest = 0`.
- Mức 2: `lowest = min(lowest, bal)` sau mỗi ký tự.
- Mức 3: `return bal - 2 * lowest`.

**Bài 5**

- Mức 1: số hoán đổi bằng `⌈extra / 2⌉` vì mỗi hoán đổi sửa hai `')'` thừa.
- Mức 2: `extra` = số `')'` thừa (bộ đếm).
- Mức 3: `(extra + 1) // 2`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [32]:
vals = (Solution().minAddToMakeValid("()))(("), Solution().minAddToMakeValid("(()())"), Solution().minAddToMakeValid("))(("), min_insertions_1541("(()))"))
assert vals == (4, 0, 4, 1) == (oracle("()))(("), oracle("(()())"), oracle("))(("), oracle_1541("(()))"))
print("Bài 1 OK:", vals)

Bài 1 OK: (4, 0, 4, 1)


### Bài 2

(a) `1, 2, 1, 0, −1, 0, 1, 0`. (b) `lowest = −1`, số dư cuối `0`. (c) Đáp án `2`; chuỗi `"(" + s + ")"` hợp lệ.

In [33]:
s = "(()))(()"
bals, b = [], 0
for ch in s:
    b += 1 if ch == "(" else -1
    bals.append(b)
assert bals == [1, 2, 1, 0, -1, 0, 1, 0]
assert min(0, min(bals)) == -1 and bals[-1] == 0
assert Solution().minAddToMakeValid(s) == oracle(s) == 2 and is_valid("(" + s + ")")
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [34]:
def counters_answer(s):
    open_ = extra = 0
    for ch in s:
        if ch == "(":
            open_ += 1
        elif open_:
            open_ -= 1
        else:
            extra += 1
    return open_ + extra


assert [counters_answer(s) for s, _ in examples] == [1, 3]
for s in tests[:400]:
    assert counters_answer(s) == oracle(s)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [35]:
class MySolutionAnswer:
    def minAddToMakeValid(self, s):
        bal = lowest = 0
        for ch in s:
            bal += 1 if ch == "(" else -1
            lowest = min(lowest, bal)
        return bal - 2 * lowest


assert all(MySolutionAnswer().minAddToMakeValid(s) == w for s, w in examples)
for s in tests[:400]:
    assert MySolutionAnswer().minAddToMakeValid(s) == oracle(s)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [36]:
def min_swaps_answer(s):
    open_ = extra = 0
    for ch in s:
        if ch == "(":
            open_ += 1
        elif open_:
            open_ -= 1
        else:
            extra += 1
    return (extra + 1) // 2


def min_swaps_brute(s):
    """BFS trên các chuỗi đạt được bằng hoán đổi hai ký tự."""
    from collections import deque as dq
    seen, q = {s}, dq([(s, 0)])
    while q:
        t, d = q.popleft()
        if is_valid(t):
            return d
        for i in range(len(t)):
            for j in range(i + 1, len(t)):
                if t[i] != t[j]:
                    u = list(t)
                    u[i], u[j] = u[j], u[i]
                    u = "".join(u)
                    if u not in seen:
                        seen.add(u)
                        q.append((u, d + 1))


assert min_swaps_answer("))((") == 1 and min_swaps_answer(")()(") == 1 and min_swaps_answer("()()") == 0
for L in range(2, 11, 2):
    for tup in itertools.product("()", repeat=L):
        s = "".join(tup)
        if s.count("(") == L // 2:
            assert min_swaps_answer(s) == min_swaps_brute(s), s
print("Bài 5 OK: khớp BFS hoán đổi trên mọi chuỗi cân bằng số lượng độ dài <= 10")

Bài 5 OK: khớp BFS hoán đổi trên mọi chuỗi cân bằng số lượng độ dài <= 10


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (ngoặc một loại → hai bộ đếm: `open_` chờ và `extra` thừa):**

```python
class Solution:
    def minAddToMakeValid(self, s: str) -> int:
        open_ = extra = 0
        for ch in s:
            if ch == "(":
                open_ += 1
            elif open_:
                open_ -= 1
            else:
                extra += 1
        return open_ + extra
```

**Checklist trước khi submit:**

- [ ] Xét **thứ tự** (không chỉ so số lượng hai loại ngoặc)?
- [ ] Cộng **cả hai**: `')'` thừa (`extra`) và `'('` dư cuối (`open_`)?
- [ ] Không `pop` ngăn xếp rỗng / không cho `open_` âm?
- [ ] Công thức thay thế: `bal − 2·min(0, min_prefix)`?
- [ ] Đã thử `"())" → 1`, `"(((" → 3`, `")(" → 2`, `"()" → 0`, một ký tự?
- [ ] Đúng tên `class Solution`, `minAddToMakeValid(self, s)`, không in debug.